# Diazepam Dose-Response Analysis of Human Brain Organoid Activity

**Companion notebook for:** *[Manuscript title]*

This notebook demonstrates a complete electrophysiology analysis pipeline using **SpikeLab**, applied to a diazepam dose-response experiment on human brain organoids recorded with a multi-electrode array (MEA).

## Dataset

- **Experiment ID:** 200123_2953
- **Preparation:** Human brain organoid on MEA
- **Conditions:** 5 diazepam concentrations — D0 (0 µM), D3 (3 µM), D10 (10 µM), D30 (30 µM), D50 (50 µM)
- **Units:** 177 per condition
- **Duration:** ~180 s per recording
- **Resolution:** 1 ms

## What you will learn

1. Loading spike data and organising it in an `AnalysisWorkspace`
2. Computing population firing rates and detecting network bursts
3. Per-unit spike metrics: firing rates, ISI distributions, spike-triggered population rate
4. Pairwise correlations: firing rate cross-correlations and spike time tiling coefficients (STTC)
5. Burst-aligned analysis: event-aligned slicing, burst-to-burst correlations, rank order
6. Dimensionality reduction: PCA on pairwise matrices and GPLVM latent states

> **Note:** All spike times in SpikeLab are in **milliseconds** unless explicitly stated otherwise.

---
# 1. Setup & Data Loading

In [ ]:
# Uncomment the line below to install SpikeLab with all optional dependencies:
# !pip install spikelab[all]

In [ ]:
%matplotlib inline

import pickle
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Core SpikeLab imports — these are re-exported at the package top level
from spikelab import (
    SpikeData,  # primary spike train data structure
    RateData,  # instantaneous firing rate matrix (U x T)
    RateSliceStack,  # event-aligned firing rate stack (U x T x S)
    SpikeSliceStack,  # event-aligned raw spike data (list of SpikeData)
    PairwiseCompMatrix,  # N x N pairwise comparison matrix
    PairwiseCompMatrixStack,  # N x N x S stack of pairwise matrices
    AnalysisWorkspace,  # two-level (namespace, key) result container
)

# Additional utilities not exported at top level
from spikelab.spikedata.utils import PCA_reduction, gplvm_state_entropy
from spikelab.spikedata.plot_utils import (
    plot_recording,  # multi-panel recording figure (raster, pop rate, FR heatmap, etc.)
    plot_distribution,  # violin / box / strip plots for distributions
    plot_heatmap,  # 2D matrix visualisation with colorbar
    plot_scatter,  # scatter plot with optional identity line and colouring
    plot_scatter_with_marginals,  # scatter with marginal histograms on x and y axes
    plot_percentile_bands,  # median + IQR band plots across conditions
    plot_burst_sensitivity,  # burst count vs threshold parameter sweep
    plot_spatial_network,  # MEA electrode layout with correlation edges
    plot_manifold,  # 2D/3D embedding visualisation (PCA, UMAP, etc.)
)

# Notebook-wide figure styling
plt.rcParams.update(
    {
        "figure.dpi": 150,  # high-resolution inline figures
        "font.size": 9,  # base font size
        "axes.titlesize": 10,  # subplot titles
        "axes.labelsize": 9,  # axis labels
        "xtick.labelsize": 8,  # x-axis tick labels
        "ytick.labelsize": 8,  # y-axis tick labels
        "legend.fontsize": 8,  # legend text
        "figure.facecolor": "white",  # white background (for dark IDE themes)
    }
)

### Download data from Zenodo

The spike-sorted data for this experiment is available as a set of pickle files on Zenodo:

> **DOI:** [10.5281/zenodo.XXXXXXX](https://doi.org/10.5281/zenodo.XXXXXXX) *(placeholder — replace with actual DOI)*

Download the five `.pkl` files and place them in a `data/` directory alongside this notebook:

```
examples/
├── manuscript_analysis.ipynb
└── data/
    ├── 200123_2953_D0.pkl
    ├── 200123_2953_D3.pkl
    ├── 200123_2953_D10.pkl
    ├── 200123_2953_D30.pkl
    └── 200123_2953_D50.pkl
```

Each file contains a serialised `SpikeData` object with 177 units and ~180 s of recording.

In [ ]:
# --- Configuration ---
DATA_DIR = Path("data")  # directory containing the pickle files
EXPERIMENT = "200123_2953"  # experiment identifier
CONDITIONS = [
    "D0",
    "D3",
    "D10",
    "D30",
    "D50",
]  # diazepam concentrations (0, 3, 10, 30, 50 µM)
LABELS = ["0 µM", "3 µM", "10 µM", "30 µM", "50 µM"]  # human-readable labels for plots
COLORS = [
    "#1f77b4",
    "#ff7f0e",
    "#2ca02c",
    "#d62728",
    "#9467bd",
]  # per-condition colours

# Load each condition from its pickle file into a dict of SpikeData objects
recordings = {}
for cond in CONDITIONS:
    pkl_path = DATA_DIR / f"{EXPERIMENT}_{cond}.pkl"  # e.g. data/200123_2953_D0.pkl
    with open(pkl_path, "rb") as f:
        recordings[cond] = pickle.load(f)  # each pickle contains one SpikeData

# Quick sanity check — print unit count, duration, and total spike count
for cond, sd in recordings.items():
    print(
        f"{cond}: {sd.N} units, {sd.length / 1000:.1f} s, "
        f"{sum(len(t) for t in sd.train)} total spikes"
    )

### Create an AnalysisWorkspace

An `AnalysisWorkspace` is a two-level `(namespace, key)` container for storing analysis results. Each condition becomes a namespace. The workspace can be saved to disk as HDF5 for reproducibility.

In [ ]:
# Create a workspace — a two-level (namespace, key) container for organising results
ws = AnalysisWorkspace(name="200123_2953_diazepam")

# Store each recording under its condition name as namespace, key "spikedata"
for cond, sd in recordings.items():
    ws.store(cond, "spikedata", sd, note=f"Raw spike data for {cond}")

# Print the workspace structure: namespaces, keys, and stored types
ws.describe()

In [ ]:
# Save the workspace to disk as an HDF5 file
ws_path = Path("workspace_200123_2953")
ws.save(ws_path)  # creates workspace_200123_2953.h5
print(f"Workspace saved to {ws_path.resolve()}")

---
# 2. Basic Spike Properties (Figure 1)

We begin by characterising the baseline (D0) recording: population firing rate, network burst detection, per-unit firing rates, interspike interval distributions, and spike-triggered population rate coupling.

### 2.1 Population rate and burst detection

The population rate is computed by binning spikes at 1 ms resolution, then smoothing with a square kernel (20 ms) followed by a Gaussian kernel (sigma = 100 ms). An "accelerated" version with narrower kernels (8 ms each) is used for precise burst peak localisation.

Bursts are detected as peaks in the accelerated population rate that exceed `thr_burst` times the RMS of the signal, with a minimum inter-burst interval of `min_burst_diff` ms. Burst edges are defined where the signal drops below `burst_edge_mult_thresh` times the peak amplitude.

In [ ]:
# Compute population rate for D0 (baseline condition)
sd = recordings["D0"]

# Display-quality population rate: broad smoothing for overview visualisation
# square_width=20: 20-sample moving average window
# gauss_sigma=100: 100 ms Gaussian smoothing kernel
# raster_bin_size_ms=1.0: bin spikes at 1 ms resolution before smoothing
pop_rate = sd.get_pop_rate(square_width=20, gauss_sigma=100, raster_bin_size_ms=1.0)

# Accurate population rate: narrow kernels for precise burst detection
# Preserves fast transients that broader smoothing would blur
pop_rate_acc = sd.get_pop_rate(square_width=8, gauss_sigma=8, raster_bin_size_ms=1.0)

print(f"Population rate: {len(pop_rate)} bins ({len(pop_rate) / 1000:.1f} s)")
print(f"Peak population rate: {pop_rate.max():.2f}")

In [ ]:
# Detect bursts
# thr_burst=2.5 means peaks must exceed 2.5x RMS of the population rate
# min_burst_diff=1000 enforces at least 1 second between burst peaks
# burst_edge_mult_thresh=0.2 sets burst boundaries at 20% of peak amplitude
tburst, burst_edges, peak_amp = sd.get_bursts(
    thr_burst=2.5,
    min_burst_diff=1000,
    burst_edge_mult_thresh=0.2,
    pop_rate=pop_rate,
    pop_rate_acc=pop_rate_acc,
)

print(f"Detected {len(tburst)} bursts in D0")
print(f"Burst peak times (s): {tburst / 1000}")
print(f"Mean burst amplitude: {peak_amp.mean():.2f}")

In [ ]:
# Plot spike raster + population rate for D0 with detected burst windows
fig = plot_recording(
    sd,
    show_raster=True,  # top panel: spike raster (one row per unit)
    show_pop_rate=True,  # bottom panel: population firing rate trace
    pop_rate=pop_rate_acc,  # use the accurate (narrow-kernel) rate for display
    burst_edges=burst_edges,  # shade detected burst windows on the pop rate panel
    raster_style="eventplot",  # vertical tick marks (vs scatter dots)
    figsize=(12, 5),
    title="D0 (0 µM diazepam) — Raster & Population Rate",
)
plt.tight_layout()
plt.show()

### 2.2 Per-unit firing metrics

We compute three per-unit metrics:
- **Firing rate** (Hz): mean spike count per second
- **Interspike intervals** (ISI): time between consecutive spikes within each unit
- **Spike-triggered population rate** (stPR): how much the population rate deviates when a given unit fires, quantifying functional coupling

In [ ]:
# Compute per-unit firing rate and ISI for all conditions
rates_per_cond = {}  # condition -> array of mean firing rates per unit (Hz)
isis_per_cond = {}  # condition -> list of ISI arrays per unit (ms)
for cond, sd in recordings.items():
    rates_per_cond[cond] = sd.rates(unit="Hz")  # mean spikes/second per unit
    isis_per_cond[cond] = sd.interspike_intervals()  # list of arrays, one per unit

# Print summary statistics for the baseline condition
rates_d0 = rates_per_cond["D0"]
print(
    f"D0 firing rates — mean: {rates_d0.mean():.2f} Hz, "
    f"median: {np.median(rates_d0):.2f} Hz, "
    f"range: [{rates_d0.min():.2f}, {rates_d0.max():.2f}] Hz"
)

In [ ]:
# Plot firing rate distributions across conditions
fig, ax = plt.subplots(figsize=(8, 4))
plot_distribution(
    ax,
    metric_data=[rates_per_cond[c] for c in CONDITIONS],
    style="violin",
    labels=LABELS,
    colors=COLORS,
    ylabel="Firing rate (Hz)",
    title="Per-unit firing rate across diazepam concentrations",
    show_data=True,
    data_alpha=0.3,
    data_size=2,
)
plt.tight_layout()
plt.show()

In [ ]:
# Plot ISI distributions (log scale) — D0 vs D50
fig, ax = plt.subplots(figsize=(8, 4))
isi_d0_all = np.concatenate(isis_per_cond["D0"])
isi_d50_all = np.concatenate(isis_per_cond["D50"])
plot_distribution(
    ax,
    metric_data=[
        np.log10(isi_d0_all[isi_d0_all > 0]),
        np.log10(isi_d50_all[isi_d50_all > 0]),
    ],
    style="violin",
    labels=["D0 (0 µM)", "D50 (50 µM)"],
    colors=[COLORS[0], COLORS[4]],
    ylabel="log10(ISI) [ms]",
    title="Interspike interval distributions — D0 vs D50",
)
plt.tight_layout()
plt.show()

In [ ]:
# Spike-triggered population rate (stPR) for D0
# Measures how each unit's firing relates to the population activity
stPR, coupling_zero, coupling_max, delays, lags = recordings[
    "D0"
].compute_spike_trig_pop_rate(
    window_ms=80,  # ±80 ms window around each spike
    cutoff_hz=20,  # low-pass filter cutoff for the population rate
    fs=1000,  # sampling rate (1 kHz = 1 ms bins)
    bin_size=1,  # 1 ms bins for spike binning
    cut_outer=10,  # exclude outer 10 ms from coupling calculation
)

# stPR shape: (units, lag_bins) — average population rate around each unit's spikes
print(f"stPR shape: {stPR.shape} (units x lag bins)")
# coupling_zero: population coupling at zero lag (synchrony measure)
print(f"Mean coupling at zero lag: {coupling_zero.mean():.4f}")
# coupling_max: peak coupling at any lag (connectivity strength)
print(f"Mean max coupling: {coupling_max.mean():.4f}")

### 2.3 Burst sensitivity analysis

Before committing to a single threshold, it is useful to sweep detection parameters and examine how the number of detected bursts varies. `burst_sensitivity` sweeps over combinations of `thr_burst` and `min_burst_diff`.

In [ ]:
# Burst sensitivity sweep for D0
thr_values = np.arange(1.0, 5.5, 0.5)
dist_values = np.array([200, 500, 1000, 2000, 5000])

burst_counts = recordings["D0"].burst_sensitivity(
    thr_values=thr_values,
    dist_values=dist_values,
    burst_edge_mult_thresh=0.2,
    pop_rate=pop_rate,
    pop_rate_acc=pop_rate_acc,
)

print(f"Burst count matrix shape: {burst_counts.shape} " f"(thresholds x distances)")

In [ ]:
# Plot burst sensitivity
fig, ax = plt.subplots(figsize=(7, 4))
plot_burst_sensitivity(
    ax,
    thresholds=thr_values,
    burst_counts=burst_counts,
    dist_values=dist_values,
    title="Burst detection sensitivity — D0",
)
plt.tight_layout()
plt.show()

### 2.4 Burst participation metrics

For each detected burst, we compute which units are active and their contribution. `get_frac_active` returns the fraction of bursts each unit participates in and the fraction of units active per burst.

In [ ]:
# Compute burst participation metrics for D0 and D50
burst_metrics = {}
for cond in ["D0", "D50"]:
    sd_c = recordings[cond]

    # Compute both population rate variants for burst detection
    pr_acc = sd_c.get_pop_rate(square_width=8, gauss_sigma=8, raster_bin_size_ms=1.0)
    pr = sd_c.get_pop_rate(square_width=20, gauss_sigma=100, raster_bin_size_ms=1.0)

    # Detect bursts with the same parameters as before
    tb, edges, _ = sd_c.get_bursts(
        thr_burst=2.5,
        min_burst_diff=1000,
        burst_edge_mult_thresh=0.2,
        pop_rate=pr,
        pop_rate_acc=pr_acc,
    )

    # Per-unit burst participation:
    # frac_per_unit: fraction of bursts in which each unit fires >= MIN_SPIKES
    # frac_per_burst: fraction of units active in each burst
    # backbone: boolean mask of units active in > backbone_threshold of bursts
    frac_per_unit, frac_per_burst, backbone = sd_c.get_frac_active(
        edges=edges,  # burst start/end times
        MIN_SPIKES=3,  # minimum spikes to count as "active" in a burst
        backbone_threshold=0.8,  # fraction of bursts a unit must be active in to be "backbone"
        bin_size=1.0,  # 1 ms bin size for edge conversion
    )

    burst_metrics[cond] = {
        "frac_per_unit": frac_per_unit,
        "frac_per_burst": frac_per_burst,
        "n_bursts": len(tb),
    }
    print(
        f"{cond}: {len(tb)} bursts, "
        f"{np.sum(backbone)} backbone units (active in >80% of bursts)"
    )

In [ ]:
# Scatter: fraction of bursts active (per unit) — D0 vs D50
fig, ax = plt.subplots(figsize=(5, 5))
plot_scatter(
    ax,
    x=burst_metrics["D0"]["frac_per_unit"],
    y=burst_metrics["D50"]["frac_per_unit"],
    xlabel="Fraction of bursts active (D0)",
    ylabel="Fraction of bursts active (D50)",
    title="Per-unit burst participation — D0 vs D50",
    fit=True,
    alpha=0.5,
    size=10,
)
# Add unity line
ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, alpha=0.5)
plt.tight_layout()
plt.show()

---
# 3. Pairwise Correlations (Figure 2)

We examine functional connectivity between units using two complementary approaches:

1. **Firing rate (FR) cross-correlation** — measures linear correlation between instantaneous firing rate time series of unit pairs at variable lags
2. **Spike Time Tiling Coefficient (STTC)** — measures co-occurrence of spike times within a coincidence window, independent of firing rate

Both produce N x N pairwise matrices stored as `PairwiseCompMatrix` objects.

### 3.1 Instantaneous firing rate and RateData

We compute instantaneous firing rates via ISI interpolation at 1 ms resolution, then wrap them in a `RateData` object for downstream analysis.

In [ ]:
# Compute instantaneous firing rates for D0 via ISI interpolation
sd = recordings["D0"]
times = np.arange(0, sd.length, 1.0)  # evaluation times at 1 ms resolution

# resampled_isi: estimates firing rate at each time point by interpolating
# the inverse ISI, then smoothing with a Gaussian kernel (sigma_ms)
fr = sd.resampled_isi(times, sigma_ms=10.0)  # shape (N_units, T_bins)

# Wrap the firing rate matrix in a RateData object for pairwise analysis methods
rd = RateData(fr, times)

print(f"RateData: {rd.N} units, {len(rd.times)} time bins")
print(f"Mean FR range: [{fr.mean(axis=1).min():.4f}, {fr.mean(axis=1).max():.4f}] kHz")

In [ ]:
# Pairwise FR cross-correlation matrix
# max_lag=350: search for peak correlation within ±350 ms lag
# Returns two PairwiseCompMatrix objects: peak correlation and corresponding lag
corr_pcm, lag_pcm = rd.get_pairwise_fr_corr(max_lag=350)

print(f"FR correlation matrix: {corr_pcm.matrix.shape}")
# extract_lower_triangle: get off-diagonal values as a flat array (avoids self-pairs)
print(
    f"Mean off-diagonal correlation: " f"{corr_pcm.extract_lower_triangle().mean():.4f}"
)

In [ ]:
# Spike Time Tiling Coefficient (STTC) — pairwise spike timing similarity
# delt=20.0: coincidence window of ±20 ms (spikes within 20 ms count as co-occurring)
# Returns a PairwiseCompMatrix with STTC values in [-1, 1]
sttc_pcm = sd.spike_time_tilings(delt=20.0)

print(f"STTC matrix: {sttc_pcm.matrix.shape}")
print(f"Mean off-diagonal STTC: " f"{sttc_pcm.extract_lower_triangle().mean():.4f}")

### 3.2 Visualising rasters, FR heatmaps, and correlation matrices

In [ ]:
# Plot spike raster with instantaneous firing rate heatmap below
fig = plot_recording(
    sd,
    show_raster=True,  # top panel: spike raster
    show_fr_rates=True,  # bottom panel: firing rate heatmap (units x time)
    fr_rates=fr,  # pre-computed instantaneous firing rates
    raster_style="eventplot",  # vertical tick marks for each spike
    figsize=(12, 6),
    title="D0 — Raster & Instantaneous Firing Rate Heatmap",
)
plt.tight_layout()
plt.show()

In [ ]:
# Plot FR correlation and STTC matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

plot_heatmap(
    corr_pcm.matrix, ax=axes[0], title="FR Correlation (D0)", cbar_label="Pearson r"
)
plot_heatmap(
    sttc_pcm.matrix, ax=axes[1], title="STTC (D0, delt=20 ms)", cbar_label="STTC"
)

plt.tight_layout()
plt.show()

In [ ]:
# Scatter: FR correlation vs STTC with marginal distributions
fig = plt.figure(figsize=(7, 7))
import matplotlib.gridspec as gridspec

gs = gridspec.GridSpec(1, 1)

plot_scatter_with_marginals(
    gs_slot=gs[0, 0],
    fig=fig,
    x=corr_pcm.extract_lower_triangle(),
    y=sttc_pcm.extract_lower_triangle(),
    xlabel="FR correlation (Pearson r)",
    ylabel="STTC",
    title="FR correlation vs STTC — D0",
    alpha=0.3,
    size=5,
    fit=True,
)
plt.tight_layout()
plt.show()

### 3.3 Correlation distributions across conditions

In [ ]:
# Compute FR correlations and STTC for all 5 conditions
fr_corr_triangles = {}  # condition -> flat array of off-diagonal FR correlations
sttc_triangles = {}  # condition -> flat array of off-diagonal STTC values

for cond, sd_c in recordings.items():
    # Instantaneous firing rate via ISI interpolation (1 ms resolution, 10 ms smoothing)
    t_c = np.arange(0, sd_c.length, 1.0)
    fr_c = sd_c.resampled_isi(t_c, sigma_ms=10.0)
    rd_c = RateData(fr_c, t_c)

    # Pairwise FR cross-correlation (search within ±350 ms)
    corr_c, _ = rd_c.get_pairwise_fr_corr(max_lag=350)
    fr_corr_triangles[cond] = corr_c.extract_lower_triangle()  # flatten upper triangle

    # Pairwise STTC (±20 ms coincidence window)
    sttc_c = sd_c.spike_time_tilings(delt=20.0)
    sttc_triangles[cond] = sttc_c.extract_lower_triangle()

    print(
        f"{cond}: mean FR corr = {fr_corr_triangles[cond].mean():.4f}, "
        f"mean STTC = {sttc_triangles[cond].mean():.4f}"
    )

In [ ]:
# Violin plots of correlation distributions across conditions
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_distribution(
    axes[0],
    metric_data=[fr_corr_triangles[c] for c in CONDITIONS],
    style="violin",
    labels=LABELS,
    colors=COLORS,
    ylabel="FR correlation (Pearson r)",
    title="Pairwise FR correlation across conditions",
)

plot_distribution(
    axes[1],
    metric_data=[sttc_triangles[c] for c in CONDITIONS],
    style="violin",
    labels=LABELS,
    colors=COLORS,
    ylabel="STTC",
    title="Pairwise STTC across conditions",
)

plt.tight_layout()
plt.show()

### 3.4 Network analysis

We convert the STTC matrix to a `networkx.Graph` by thresholding, then detect functional communities using the Louvain algorithm.

In [ ]:
import networkx as nx
import community as community_louvain  # python-louvain package

# Convert STTC matrix to a networkx graph, keeping only edges above threshold
sttc_d0 = recordings["D0"].spike_time_tilings(delt=20.0)
G = sttc_d0.to_networkx(threshold=0.1)  # edges with STTC > 0.1

print(f"Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Louvain community detection — finds groups of functionally connected units
partition = community_louvain.best_partition(G, random_state=42)
n_communities = len(set(partition.values()))
print(f"Louvain detected {n_communities} communities")

# Map partition dict to an array indexed by unit number
community_labels = np.array([partition.get(i, -1) for i in range(recordings["D0"].N)])

---
# 4. Burst Dynamics (Figure 3)

Event-aligned analysis reveals the internal structure of network bursts. We align all data to burst peak times and examine:
- Single-unit rasters across bursts
- Burst-to-burst firing rate correlations (stereotypy)
- Unit-to-unit correlations within bursts
- PCA on pairwise correlation structure
- Rank order of unit activation across bursts

### 4.1 Event-aligned slicing

`align_to_events` extracts windows around each burst peak, producing either a `RateSliceStack` (smoothed firing rates, shape U x T x S) or a `SpikeSliceStack` (list of SpikeData objects, one per burst).

In [ ]:
# Create burst-aligned stacks for D0
sd = recordings["D0"]

# Re-detect bursts (reuse parameters from Section 2)
pop_rate_acc_d0 = sd.get_pop_rate(square_width=8, gauss_sigma=8, raster_bin_size_ms=1.0)
pop_rate_d0 = sd.get_pop_rate(square_width=20, gauss_sigma=100, raster_bin_size_ms=1.0)
tburst_d0, edges_d0, _ = sd.get_bursts(
    thr_burst=2.5,
    min_burst_diff=1000,
    burst_edge_mult_thresh=0.2,
    pop_rate=pop_rate_d0,
    pop_rate_acc=pop_rate_acc_d0,
)

# RateSliceStack: firing rate matrix centred on each burst peak
# pre_ms=250: include 250 ms before the burst peak
# post_ms=500: include 500 ms after the burst peak
# kind="rate": compute instantaneous firing rates (not raw spikes)
# sigma_ms=10: Gaussian smoothing kernel for the rate estimate
rss = sd.align_to_events(
    tburst_d0,
    pre_ms=250,
    post_ms=500,
    kind="rate",
    bin_size_ms=1.0,
    sigma_ms=10,
)
print(
    f"RateSliceStack shape: {rss.event_stack.shape} (U x T x S)"
)  # units x time x slices

# SpikeSliceStack: raw spike trains centred on each burst peak
# Same time window, but preserves individual spike times instead of rates
sss = sd.align_to_events(
    tburst_d0,
    pre_ms=250,
    post_ms=500,
    kind="spike",  # returns SpikeSliceStack (list of SpikeData objects)
)
print(f"SpikeSliceStack: {len(sss.spike_stack)} slices, " f"{sss.N} units each")

In [ ]:
# Plot single-unit burst raster (unit 0 across all bursts)
fig, ax = plt.subplots(figsize=(8, 4))
sss.plot_aligned_slice_single_unit(unit_idx=0, ax=ax)
ax.set_title("Unit 0 — spike raster across burst events (D0)")
ax.set_xlabel("Time from burst peak (ms)")
ax.set_ylabel("Burst index")
plt.tight_layout()
plt.show()

### 4.2 Burst-to-burst correlations (stereotypy)

We measure how similar each pair of bursts is by computing per-unit correlations across burst windows. High burst-to-burst correlations indicate stereotyped network activation patterns.

In [ ]:
# Burst-to-burst (slice-to-slice) correlations
# For each unit, correlates its firing rate profile across all pairs of bursts.
# Returns a PairwiseCompMatrixStack of shape (S, S, U) and the unit-averaged matrix (S, S).
b2b_stack, av_b2b = rss.get_slice_to_slice_unit_corr_from_stack()

print(f"Burst-to-burst correlation stack: {b2b_stack.stack.shape} (S x S x U)")
print(f"Mean per-unit burst-to-burst correlation: {av_b2b.mean():.4f}")

In [ ]:
# Plot mean burst-to-burst correlation matrix (averaged across units)
mean_b2b = b2b_stack.mean()

fig, ax = plt.subplots(figsize=(5, 5))
plot_heatmap(
    mean_b2b.matrix,
    ax=ax,
    title="Mean burst-to-burst correlation (D0)",
    cbar_label="Pearson r",
)
ax.set_xlabel("Burst index")
ax.set_ylabel("Burst index")
plt.tight_layout()
plt.show()

### 4.3 Unit-to-unit correlations within bursts + PCA

Within each burst window, we compute pairwise unit-to-unit firing rate correlations. The lower triangle of each burst's correlation matrix is extracted as a feature vector, then PCA is applied to visualise the burst-level correlation structure across conditions.

In [ ]:
# Unit-to-unit correlations within each burst window
# For each burst slice, computes the pairwise correlation between all unit pairs.
# max_lag=10: search for peak cross-correlation within ±10 ms
# Returns: correlation stack (U,U,S), lag stack (U,U,S), and per-slice averages
u2u_corr_stack, u2u_lag_stack, av_corr, av_lag = rss.unit_to_unit_correlation(
    max_lag=10
)

print(f"Unit-to-unit correlation stack: {u2u_corr_stack.stack.shape} (U x U x S)")

# Extract the lower triangle of each U×U matrix and stack as feature vectors
# Result shape: (S, F) where F = U*(U-1)/2 unique pairs — one row per burst
features = u2u_corr_stack.extract_lower_triangle_features()
print(f"Feature matrix: {features.shape} (bursts x pairwise features)")

In [ ]:
# PCA on burst correlation features (D0 only)
# NaN values arise when a unit has no spikes in a burst — replace with 0 (uncorrelated)
features_clean = np.nan_to_num(features, nan=0.0)

# PCA_reduction: sklearn PCA wrapper returning (embedding, variance_ratio)
embedding, var_ratio = PCA_reduction(features_clean, n_components=3)

print(f"PCA embedding: {embedding.shape}")  # (n_bursts, 3)
print(
    f"Explained variance: PC1={var_ratio[0]:.2%}, PC2={var_ratio[1]:.2%}, "
    f"PC3={var_ratio[2]:.2%}"
)

In [ ]:
# Multi-condition PCA: compute burst u2u features for ALL conditions, then project jointly
all_features = []  # list of feature matrices, one per condition
cond_idx = []  # integer label per burst (0=D0, 1=D3, ...)

for i, cond in enumerate(CONDITIONS):
    sd_c = recordings[cond]

    # Detect bursts for this condition
    pr_acc_c = sd_c.get_pop_rate(square_width=8, gauss_sigma=8, raster_bin_size_ms=1.0)
    pr_c = sd_c.get_pop_rate(square_width=20, gauss_sigma=100, raster_bin_size_ms=1.0)
    tb_c, _, _ = sd_c.get_bursts(
        thr_burst=2.5,
        min_burst_diff=1000,
        burst_edge_mult_thresh=0.2,
        pop_rate=pr_c,
        pop_rate_acc=pr_acc_c,
    )
    if len(tb_c) == 0:
        print(f"{cond}: no bursts detected, skipping")
        continue

    # Burst-aligned firing rates for this condition
    rss_c = sd_c.align_to_events(
        tb_c,
        pre_ms=250,
        post_ms=500,
        kind="rate",
        bin_size_ms=1.0,
        sigma_ms=10,
    )

    # Unit-to-unit correlations within each burst, then extract pairwise features
    u2u_c, _, _, _ = rss_c.unit_to_unit_correlation(max_lag=10)
    feat_c = u2u_c.extract_lower_triangle_features()  # (S_cond, F)

    all_features.append(feat_c)
    cond_idx.extend([i] * feat_c.shape[0])  # label each burst with its condition
    print(f"{cond}: {feat_c.shape[0]} bursts")

# Stack all conditions and fill NaNs
all_features = np.nan_to_num(np.vstack(all_features), nan=0.0)
cond_idx = np.array(cond_idx)

# Joint PCA across all conditions — reveals how burst structure changes with diazepam
embedding_all, var_ratio_all = PCA_reduction(all_features, n_components=3)
print(
    f"\nJoint PCA: {embedding_all.shape[0]} bursts, "
    f"explained variance = {var_ratio_all[:3].sum():.2%}"
)

In [ ]:
# Plot burst PCA coloured by condition
fig, ax = plt.subplots(figsize=(7, 6))
plot_scatter(
    ax,
    x=embedding_all[:, 0],
    y=embedding_all[:, 1],
    color_vals=cond_idx,
    xlabel=f"PC1 ({var_ratio_all[0]:.1%})",
    ylabel=f"PC2 ({var_ratio_all[1]:.1%})",
    title="Burst correlation PCA — all conditions",
    alpha=0.7,
    size=30,
)

# Add legend
for i, label in enumerate(LABELS):
    ax.scatter([], [], c=plt.cm.viridis(i / (len(LABELS) - 1)), label=label, s=30)
ax.legend(loc="best", framealpha=0.8)
plt.tight_layout()
plt.show()

### 4.4 Slice-to-slice time correlations and rank order

We also examine how the temporal profile of each burst (across all units at each time bin) correlates across bursts, and whether units activate in a consistent rank order.

In [ ]:
# Slice-to-slice time correlations
# For each pair of bursts, correlates the population activity at each time bin.
# Returns a stack of shape (S, S, T) and the time-averaged correlation (T,).
time_corr_stack, av_time_corr = rss.get_slice_to_slice_time_corr_from_stack()

print(f"Time correlation stack: {time_corr_stack.stack.shape} (S x S x T)")
print(f"Mean time-bin correlation across bursts: {av_time_corr.mean():.4f}")

In [ ]:
# Rank order analysis — are units activated in the same sequence across bursts?

# Get the time bin of peak firing rate for each unit in each burst
# MIN_RATE_THRESHOLD=0.1: units with peak rate below this are marked NaN (inactive)
timing_matrix = rss.get_unit_timing_per_slice(MIN_RATE_THRESHOLD=0.1)
print(f"Timing matrix: {timing_matrix.shape} (U x S)")  # units x bursts

# Pairwise Spearman rank-order correlation across all burst pairs
# n_shuffles=100: shuffle unit order 100 times to compute z-scores
# seed=42: reproducible shuffle randomisation
rank_corr, av_rank, rank_shuffled = rss.rank_order_correlation(
    timing_matrix=timing_matrix,
    n_shuffles=100,
    seed=42,
)

print(f"Rank order correlation matrix: {rank_corr.matrix.shape} (S x S)")
print(f"Mean rank order correlation: {av_rank:.4f}")

In [ ]:
# Plot rank order correlation heatmap
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

plot_heatmap(
    rank_corr.matrix,
    ax=axes[0],
    title="Rank order correlation (D0)",
    cbar_label="Spearman rho",
)
axes[0].set_xlabel("Burst index")
axes[0].set_ylabel("Burst index")

plot_heatmap(
    rank_shuffled.matrix,
    ax=axes[1],
    title="Rank order z-score vs shuffle (D0)",
    cbar_label="z-score",
)
axes[1].set_xlabel("Burst index")
axes[1].set_ylabel("Burst index")

plt.tight_layout()
plt.show()

---
# 5. GPLVM Latent State Analysis (Figure 4) — Optional

The Gaussian Process Latent Variable Model (GPLVM) fits a low-dimensional latent trajectory to the population spike activity. This requires the optional dependencies `poor_man_gplvm` and `jax`.

The approach:
1. Stitch burst windows into a continuous sequence (with 100 ms gaps)
2. Fit a 1D GPLVM to the stitched spike counts
3. Decode latent states and visualise the state trajectory
4. Compute state entropy as a summary measure of complexity

> **Note:** If JAX is not installed, this section will be skipped gracefully.

In [ ]:
# Check for JAX availability
HAS_JAX = False
try:
    import jax
    import poor_man_gplvm

    HAS_JAX = True
    print(f"JAX version: {jax.__version__}")
    print("GPLVM dependencies available — proceeding with Section 5")
except ImportError:
    print("JAX or poor_man_gplvm not installed — Section 5 will be skipped")
    print("Install with: pip install jax poor_man_gplvm")

In [ ]:
if HAS_JAX:
    # Stitch burst windows into a single continuous SpikeData
    # Each burst is a SpikeData from the SpikeSliceStack
    burst_slices = sss.spike_stack  # list of SpikeData, one per burst

    # Concatenate bursts with 100 ms silence gaps between them
    stitched = burst_slices[0]
    for burst_sd in burst_slices[1:]:
        stitched = stitched.append(burst_sd, offset=100)  # 100 ms gap

    print(f"Stitched recording: {stitched.N} units, {stitched.length / 1000:.1f} s")

    # Fit GPLVM — discovers latent states from the population activity
    # bin_size_ms=50: temporal resolution for binning spikes before fitting
    # n_latent_bin=100: number of latent state bins
    # n_iter=20: optimisation iterations
    gplvm_result = stitched.fit_gplvm(
        bin_size_ms=50,
        n_latent_bin=100,
        n_iter=20,
        random_seed=42,
    )

    decode_res = gplvm_result["decode_res"]  # posterior state probabilities
    reorder = gplvm_result["reorder_indices"]  # unit ordering from GPLVM
    print(
        f"GPLVM fitted: {decode_res['posterior_latent_marg'].shape[0]} time bins, "
        f"{decode_res['posterior_latent_marg'].shape[1]} latent bins"
    )
else:
    print("Skipping GPLVM fitting (JAX not available)")

In [ ]:
if HAS_JAX:
    # Plot stitched burst raster with GPLVM-decoded latent states overlaid
    fig = plot_recording(
        stitched,
        show_raster=True,  # spike raster panel
        show_model_states=True,  # GPLVM state probability heatmap panel
        model_states=decode_res,  # posterior state probabilities from fit_gplvm
        raster_style="eventplot",
        figsize=(14, 6),
        title="Stitched bursts (D0) — Raster & GPLVM latent states",
    )
    plt.tight_layout()
    plt.show()

    # State entropy: how uncertain the GPLVM is about the current state
    # High entropy = distributed across states; low = confidently in one state
    entropy = gplvm_state_entropy(decode_res["posterior_latent_marg"])
    print(f"GPLVM state entropy: {entropy:.4f} bits")
else:
    print("Skipping GPLVM visualisation (JAX not available)")

### 5.1 Rate PCA on the full recording

Independent of the GPLVM, we can also apply PCA directly to the instantaneous firing rate matrix via `RateData.get_manifold()`. This provides a complementary low-dimensional view of population dynamics.

In [ ]:
# Rate PCA — dimensionality reduction on the full instantaneous firing rate trace
sd = recordings["D0"]
times_d0 = np.arange(0, sd.length, 1.0)  # 1 ms time points
fr_d0 = sd.resampled_isi(times_d0, sigma_ms=10.0)  # ISI-interpolated rates (N x T)
rd_d0 = RateData(fr_d0, times_d0)  # wrap in RateData for manifold methods

# get_manifold: PCA on the transposed rate matrix (time points as observations)
# Returns: embedding (T x n_components), variance ratio, component loadings
embedding_pca, var_ratio_pca, components_pca = rd_d0.get_manifold(
    method="PCA",  # could also be "UMAP" with the ml extra installed
    n_components=3,  # project onto 3 principal components
)

print(f"Rate PCA embedding: {embedding_pca.shape} (time bins x components)")
print(
    f"Explained variance: PC1={var_ratio_pca[0]:.2%}, "
    f"PC2={var_ratio_pca[1]:.2%}, PC3={var_ratio_pca[2]:.2%}"
)

In [ ]:
# Plot rate PCA manifold — each point is one time bin, coloured by recording time
fig, ax = plt.subplots(figsize=(7, 6))
plot_manifold(
    ax,
    embedding_pca[:, :2],  # use first two PCs
    color_values=times_d0 / 1000,  # colour by time in seconds
    xlabel=f"PC1 ({var_ratio_pca[0]:.1%})",
    ylabel=f"PC2 ({var_ratio_pca[1]:.1%})",
    title="Rate PCA — D0 (coloured by time)",
    cbar_label="Time (s)",
    alpha=0.3,  # transparency for overlapping points
    size=1,  # small point size (many time bins)
)
plt.tight_layout()
plt.show()

---
# 6. Saving Results

All intermediate results can be stored in the workspace and persisted to disk. The workspace uses HDF5 for efficient storage and supports lazy loading for large datasets.

In [ ]:
# Store key analysis results in the workspace for later retrieval
ws.store("D0", "pop_rate", pop_rate, note="Smoothed population rate (sq=20, gauss=100)")
ws.store("D0", "burst_times", tburst_d0, note="Burst peak times (ms)")
ws.store("D0", "burst_edges", edges_d0, note="Burst edge boundaries (ms)")
ws.store("D0", "fr_corr_matrix", corr_pcm, note="Pairwise FR correlation (max_lag=350)")
ws.store("D0", "sttc_matrix", sttc_d0, note="Pairwise STTC (delt=20 ms)")
ws.store("D0", "rank_order_corr", rank_corr, note="Burst rank order correlation")

# Store per-condition firing rates
for cond in CONDITIONS:
    ws.store(
        cond, "firing_rates_hz", rates_per_cond[cond], note="Per-unit firing rates (Hz)"
    )

# Persist everything to disk
ws.save(ws_path)
print(f"Workspace saved with {len(ws.list_namespaces())} namespaces")
ws.describe()

In [ ]:
# Demonstrate loading a saved workspace from disk
ws2 = AnalysisWorkspace.load(ws_path)

# List all namespaces (conditions) in the loaded workspace
print("Loaded workspace namespaces:", ws2.list_namespaces())

# List all keys stored under the D0 namespace
print("\nD0 keys:", ws2.list_keys("D0"))

# Retrieve a specific stored array and verify it matches
loaded_rates = ws2.get("D0", "firing_rates_hz")
print(
    f"\nLoaded D0 firing rates: {loaded_rates.shape}, "
    f"mean = {loaded_rates.mean():.2f} Hz"
)

---

## Summary

This notebook demonstrated the SpikeLab analysis pipeline on a diazepam dose-response dataset:

| Section | Key methods used |
|---|---|
| **Data loading** | `SpikeData` (pickle), `AnalysisWorkspace.store()`, `.save()`, `.load()` |
| **Population rate & bursts** | `get_pop_rate()`, `get_bursts()`, `burst_sensitivity()`, `get_frac_active()` |
| **Per-unit metrics** | `rates()`, `interspike_intervals()`, `compute_spike_trig_pop_rate()` |
| **Pairwise correlations** | `resampled_isi()`, `RateData.get_pairwise_fr_corr()`, `spike_time_tilings()` |
| **Network analysis** | `PairwiseCompMatrix.to_networkx()`, Louvain community detection |
| **Burst dynamics** | `align_to_events()`, `RateSliceStack`, `SpikeSliceStack` |
| **Burst stereotypy** | `get_slice_to_slice_unit_corr_from_stack()`, `rank_order_correlation()` |
| **Burst PCA** | `unit_to_unit_correlation()`, `extract_lower_triangle_features()`, `PCA_reduction()` |
| **GPLVM** | `fit_gplvm()`, `gplvm_state_entropy()` |
| **Rate manifold** | `RateData.get_manifold(method="PCA")`, `plot_manifold()` |
| **Plotting** | `plot_recording()`, `plot_distribution()`, `plot_heatmap()`, `plot_scatter()`, `plot_scatter_with_marginals()`, `plot_burst_sensitivity()`, `plot_manifold()` |

For the full API reference, see the [SpikeLab documentation](https://spikelab.readthedocs.io).